# Bagging Model

In [1]:
# 0) Import packages
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# 1) Load the data
TRAIN_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/train.csv"
TEST_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/test.csv"
SAMPLE_SUB_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/sample_submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = -1

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [2]:
# 2) Split features / target
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].copy()

X = X.drop(columns=[ID_COL], errors="ignore")
X_test = test_df.drop(columns=[ID_COL], errors="ignore")

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [3]:
# 3) Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

In [4]:
# 4) Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [5]:
# 5) Random Forest pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        class_weight="balanced_subsample"
    ))
])

param_dist = {
    "model__n_estimators": [150, 250, 350],
    "model__max_depth": [8, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.6, 0.8],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=4,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

In [6]:
# 6) Train
search.fit(X, y_encoded)

print("Best Random Forest params:")
print(search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 5))

best_model = search.best_estimator_

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best Random Forest params:
{'model__n_estimators': 150, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 0.6, 'model__max_depth': 16}
Best CV accuracy: 0.98545


In [7]:
# 7) Fit on full train and predict test
best_model.fit(X, y_encoded)
test_pred_encoded = best_model.predict(X_test)
test_pred = label_encoder.inverse_transform(test_pred_encoded)

In [8]:
# 8) Save submission
submission = sample_sub.copy()
submission[TARGET] = test_pred
submission.to_csv("submission_bagging.csv", index=False)